# 第6章 固有表現認識(Named Entity Recognition)

## 6.2 データセット・前処理・評価指標

#### 準備

In [1]:
!pip install 'datasets<4.0.0' 'transformers[ja,torch]<4.41.0'  spacy-alignments seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 85.7 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 20.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 127.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.2 MB/s eta 0:00:00
  

In [2]:
%pip uninstall -y peft

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1


### 6.2.1 データセット

#### データセットのダウンロード

### 

In [3]:
from datasets import load_dataset

dataset = load_dataset("llm-book/ner-wikipedia-dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ner-wikipedia-dataset.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
from pprint import pprint

pprint(dataset)

DatasetDict({
    train: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 4274
    })
    validation: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 534
    })
    test: Dataset({
        features: ['curid', 'text', 'entities'],
        num_rows: 535
    })
})


In [5]:
pprint(dataset["train"]["text"][0:10])

['さくら学院、Ciao Smilesのメンバー。',
 '2008年10月5日、アウェーでのレクレアティーボ・ウェルバ戦でプリメーラ・ディビシオンでの初得点を決めた。',
 'ロシア政府に近いとされ、ロシアによるSNSを用いた世論操作を行った「オリギノのトロール工場」としても知られる。',
 '宮崎市立佐土原中学校は、宮崎県宮崎市佐土原町にある公立中学校。',
 'この時点で鈴屋は日本国内に約200店舗を有していたが、これらの事業は和議申請後にも継続されるとされた。',
 '高山祭は、岐阜県高山市で毎年開催される、4月14~15日の日枝神社例祭「春の山王祭」と、10月9~10日の櫻山八幡宮例祭「秋の八幡祭」の総称である。',
 '著名人では俳優のジョニー・デップが購入して所有している。',
 '1999年2月に日立造船因島工場においてヘリコプター格納庫内に実習用講堂を設置し、その他練習艦種別変更工事を行う。',
 'そこで、陳腐化した軽戦車や、鹵獲した戦車・車輌を改造するか、または旧型戦車の生産ラインを対戦車自走砲に切り替えるという決定がなされた。',
 '優等列車であるが、一部区間では各駅に停車となる列車種別で、1938年、京阪電鉄京阪本線に設定された区間急行が最初といわれている。']


In [6]:
pprint(list(dataset["train"])[:2])

[{'curid': '3638038',
  'entities': [{'name': 'さくら学院', 'span': [0, 5], 'type': 'その他の組織名'},
               {'name': 'Ciao Smiles', 'span': [6, 17], 'type': 'その他の組織名'}],
  'text': 'さくら学院、Ciao Smilesのメンバー。'},
 {'curid': '1729527',
  'entities': [{'name': 'レクレアティーボ・ウェルバ', 'span': [17, 30], 'type': 'その他の組織名'},
               {'name': 'プリメーラ・ディビシオン', 'span': [32, 44], 'type': 'その他の組織名'}],
  'text': '2008年10月5日、アウェーでのレクレアティーボ・ウェルバ戦でプリメーラ・ディビシオンでの初得点を決めた。'}]


#### データセットの分析

In [7]:
from collections import Counter
import pandas as pd
from datasets import Dataset

def count_label_occurrences(dataset: Dataset) -> dict[str, int]:
    """固有表現タイプの出現回数をカウント"""
    # 各事例から固有表現タイプを抽出したlistを作成する
    entities = [
        e["type"] for data in dataset for e in data["entities"]
    ]

    # ラベルの出現回数が多い順に並び替える
    label_counts = dict(Counter(entities).most_common())
    return label_counts

In [8]:
label_counts_dict = {}
for split in dataset:  # 各分割セットを処理する
    label_counts_dict[split] = count_label_occurrences(dataset[split])

# DataFrame形式で表示する
df = pd.DataFrame(label_counts_dict)
df.loc["total"] = df.sum()
display(df)

,train,validation,test
人名,2394,299,287
法人名,2006,231,248
地名,1769,184,204
政治的組織名,953,121,106
製品名,934,123,158
施設名,868,103,137
その他の組織名,852,99,100
イベント名,831,85,93
total,10607,1245,1333


#### スパンの重なる固有表現の存在を判定

In [9]:
def has_overlap(spans: list[tuple[int, int]]) -> int:
    """スパンの重なる固有表現の存在を判定"""
    # スパンを開始位置で昇順に並び替える
    sorted_spans = sorted(spans, key=lambda x: x[0])
    for i in range(1, len(sorted_spans)):
        # 前のスパンの終了位置が現在の開始位置よりも大きい場合、重なっているとする
        if sorted_spans[i-1][1] > sorted_spans[i][0]:
            return 1
    return 0

In [10]:
# 各分割セットでスパンの重なる固有表現がある事例数を数える
overlap_count = 0
for split in dataset:  # 各分割セットを処理する
    for data in dataset[split]:  # 各事例数を処理する
        if data["entities"]:  # 固有表現の存在しない事例はスキップする
            # スパンのみのlistを作成する
            spans = [e["span"] for e in data["entities"]]
            overlap_count += has_overlap(spans)
    print(f"{split}におけるスパンが重複する事例数: {overlap_count}")

trainにおけるスパンが重複する事例数: 0
validationにおけるスパンが重複する事例数: 0
testにおけるスパンが重複する事例数: 0


### 6.2.2 前処理

#### テキストの正規化

In [11]:
from unicodedata import normalize

# テキストに対してUnicode正規化を行う
text = "ＡＢＣABCａｂｃabcｱｲｳアイウ①②③123"
normalized_text = normalize("NFKC", text)
print(f"正規化前: {text}")
print(f"正規化後: {normalized_text}")

正規化前: ＡＢＣABCａｂｃabcｱｲｳアイウ①②③123
正規化後: ABCABCabcabcアイウアイウ123123


In [ ]:
text = "㈱、3㌕、10℃"
normalized_text = normalize("NFKC", text)
print(f"正規化前: {text}")
print(f"正規化後: {normalized_text}")

正規化前: ㈱、3㌕、10℃
正規化後: (株)、3キログラム、10°C


In [13]:
from unicodedata import is_normalized

count = 0
for split in dataset:
    for data in dataset[split]:
        # テキストが正規化されていない事例数をカウントする
        if not is_normalized("NFKC", data["text"]):
            count += 1
print(f"正規化されていない事例数: {count}")

正規化されていない事例数: 0


In [14]:
for split in dataset:
    pprint(split)
    for data in dataset[split]:
        pprint(data["text"])

'train'
'さくら学院、Ciao Smilesのメンバー。'
'2008年10月5日、アウェーでのレクレアティーボ・ウェルバ戦でプリメーラ・ディビシオンでの初得点を決めた。'
'ロシア政府に近いとされ、ロシアによるSNSを用いた世論操作を行った「オリギノのトロール工場」としても知られる。'
'宮崎市立佐土原中学校は、宮崎県宮崎市佐土原町にある公立中学校。'
'この時点で鈴屋は日本国内に約200店舗を有していたが、これらの事業は和議申請後にも継続されるとされた。'
'高山祭は、岐阜県高山市で毎年開催される、4月14~15日の日枝神社例祭「春の山王祭」と、10月9~10日の櫻山八幡宮例祭「秋の八幡祭」の総称である。'
'著名人では俳優のジョニー・デップが購入して所有している。'
'1999年2月に日立造船因島工場においてヘリコプター格納庫内に実習用講堂を設置し、その他練習艦種別変更工事を行う。'
'そこで、陳腐化した軽戦車や、鹵獲した戦車・車輌を改造するか、または旧型戦車の生産ラインを対戦車自走砲に切り替えるという決定がなされた。'
'優等列車であるが、一部区間では各駅に停車となる列車種別で、1938年、京阪電鉄京阪本線に設定された区間急行が最初といわれている。'
'1838年に曽国藩が湖南省試を受験したときの試験官を務めた。'
'金斗鎔は、労働者独自の運動に戻り、「労働者は祖国を持たない」という「共産党宣言」に基づいて民族的闘争を放棄することを主張した。'
'長野原支店では2018年冬期から毎年冬期の繁忙期輸送にジェイアールバス東北青森支店より新幹線E5系はやぶさカラーのハイデッカー車2台を借用して志賀草津高原線の臨時急行便で運行している。'
'同年マーカーはミルドレッド・コリンと結婚した。'
'日米地位協定というように、国家間の外交においても、条約等により両国間の役割、立場、権利、相手国及びその国民への処遇を規定する際、両国の立場を表す概念として地位と称される。'
('また、F3のメジャーイベントであるマスターズF3の大会スポンサーとなったり、フランスのミニテルにおいて「Marlboro Racing '
 'Service」と呼ばれるモータースポーツ情報の提供サービスを行っていたこともある。')
'ブリティッシュ・ホーム・チャンピオンシップは、イギリス国内の

#### テキストのトークナイゼーション

In [15]:
from transformers import AutoTokenizer

model_name = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

subwords = "/".join(tokenizer.tokenize(dataset["train"][0]["text"]))
characters = "/".join(dataset["train"][0]["text"])
print(f"サブワード単位: {subwords}")
print(f"文字単位: {characters}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

サブワード単位: さくら/学院/、/C/##ia/##o/Sm/##ile/##s/の/メンバー/。
文字単位: さ/く/ら/学/院/、/C/i/a/o/ /S/m/i/l/e/s/の/メ/ン/バ/ー/。


#### 文字列とトークン列のアライメント

In [16]:
text = "さくら学院"

In [ ]:
from spacy_alignments.tokenizations import get_alignments

# 文字列のlistを獲得する
characters = list(text)
# テキストを特殊トークンを含めたトークンのlistに変換する
tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
# 文字列のlistとトークンのlistのアライメントを取る
char_to_token_indices, token_to_char_indices = get_alignments(characters, tokens)

# 確認
print(f"文字のlist: {characters}")
print(f"文字に対するトークンの位置: {char_to_token_indices}")

print(f"トークンのlist: {tokens}")
print(f"トークンに対する文字の位置: {token_to_char_indices}")

文字のlist: ['さ', 'く', 'ら', '学', '院']
文字に対するトークンの位置: [[1], [1], [1], [2], [2]]
トークンのlist: ['[CLS]', 'さくら', '学院', '[SEP]']
トークンに対する文字の位置: [[], [0, 1, 2], [3, 4], []]


#### 系列ラベリングのためのラベル作成

In [ ]:
text = "大谷翔平は岩手県水沢市出身"
entities = [
    {"name": "大谷翔平", "span": [0,4], "type": "人名"},
    {"name": "岩手県水沢市", "span": [5,11], "type": "地名"},
]

In [19]:
from transformers import PreTrainedTokenizer

def output_tokens_and_labels(
    text: str,
    entities: list[dict[str, list[int] | str]],
    tokenizer: PreTrainedTokenizer,
) -> tuple[list[str], list[str]]:
    """トークンのlistとラベルのlistを出力"""
    # 文字のlistとトークンのlistのアライメントを取る
    characters = list(text)
    tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    char_to_token_indices, _ = get_alignments(characters, tokens)

    # "0"のラベルで初期化したラベルのlistを作成する
    labels = ["O"] * len(tokens)
    for entity in entities:  # 各固有表現で処理する
        entity_span, entity_type = entity["span"], entity["type"]
        start = char_to_token_indices[entity_span[0]][0]
        end = char_to_token_indices[entity_span[1] - 1][0]
        # 固有表現の開始トークンの位置に"B-"のラベルを設定する
        labels[start] = f"B-{entity_type}"
        # 固有表現の開始トークン以外の位置に"B-"のラベルを設定する
        for idx in range(start+1, end+1):
            labels[idx] = f"I-{entity_type}"
    # 特殊トークンの位置にはラベルを設定しない
    labels[0] = "-"   # [CLS]
    labels[-1] = "-"  # [SEP]
    return tokens, labels

In [ ]:
# トークンとラベルのlistを出力する
tokens, labels = output_tokens_and_labels(text, entities, tokenizer)
# DataFrameの形式で表示する
df = pd.DataFrame({"トークン列": tokens, "ラベル列": labels})
df.index.name = "位置"
display(df.T)

位置,0,1,2,3,4,5,6,7,8,9,10
トークン列,[CLS],大谷,翔,##平,は,岩手,県,水沢,市,出身,[SEP]
ラベル列,-,B-人名,I-人名,I-人名,O,B-地名,I-地名,I-地名,I-地名,O,-


## 6.2.3 評価指標

#### seqevalライブラリを用いた評価スコアの算出

In [21]:
from typing import Any
from seqeval.metrics import classification_report

def create_character_labels(
    text: str,
    entities: list[dict[str, list[int] | str]]
) -> list[str]:
    """文字ベースでラベルのlistを作成"""
    # "o"のラベルで初期化したラベルのlistを作成する
    labels = ["O"] * len(text)
    for entity in entities:  # 各固有表現を処理する
        entity_span, entity_type = entity["span"], entity["type"]
        # 固有表現の開始トークンの位置に"B-"のラベルを設定する
        labels[entity_span[0]] = f"B-{entity_type}"
        # 固有表現の開始トークン以外の位置に"B-"のラベルを設定する
        for i in range(entity_span[0]+1, entity_span[1]):
            labels[i] = f"I-{entity_type}"
    return labels

def convert_results_to_labels(
    results: list[dict[str, Any]]
) -> tuple[list[list[str]], list[list[str]]]:
    """正解データと予測データのラベルのlistを作成"""
    true_labels, pred_labels = [], []
    for result in results:  # 各事例を処理する
        # 文字ベースでラベルのリストを作成してlistに加える
        true_labels.append(
            create_character_labels(result["text"], result["entities"])
        )
        pred_labels.append(
            create_character_labels(result["text"], result["pred_entities"])
        )
    return true_labels, pred_labels

In [22]:
results = [
    {
        "text": "大谷翔平は岩手県水沢市出身",
        "entities": [
            {"name": "大谷翔平", "span": [0, 4], "type": "人名"},
            {"name": "岩手県水沢市", "span": [5, 11], "type": "地名"},
        ],
        "pred_entities": [
            {"name": "大谷翔平", "span": [0, 4], "type": "人名"},
            {"name": "岩手県", "span": [5, 8], "type": "地名"},
            {"name": "水沢市", "span": [8, 11], "type": "施設名"},
        ],
    }
]

# 正解データと予測データのラベルのlistを作成する
true_labels, pred_labels = convert_results_to_labels(results)
# 評価結果を取得して表示する
print(classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

          人名       1.00      1.00      1.00         1
          地名       0.00      0.00      0.00         1
         施設名       0.00      0.00      0.00         0

   micro avg       0.33      0.50      0.40         2
   macro avg       0.33      0.33      0.33         2
weighted avg       0.50      0.50      0.50         2



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_scores(
    true_labels: list[list[str]],
    pred_labels: list[list[str]],
    average: str
) -> dict[str, float]:
    """適合率、再現率、F値を算出"""
    scores = {
        "precision": precision_score(true_labels, pred_labels, average=average),
        "recall": recall_score(true_labels, pred_labels, average=average),
        "f1-score": f1_score(true_labels, pred_labels, average=average)
    }
    return scores

In [24]:
# 適合率、再現率、F値のマイクロ平均を算出する
print(compute_scores(true_labels, pred_labels, "micro"))

{'precision': np.float64(0.3333333333333333), 'recall': np.float64(0.5), 'f1-score': np.float64(0.4)}


## 6.3 固有表現認識モデルの実装

### 6.3.1 BERTのファインチューニング

#### ラベルとIDを対応付けるdictの作成

In [ ]:
import torch

def create_label2id(
    entities_list: list[list[dict[str, str | int]]]
) -> dict[str, int]:
    """ラベルとIDを紐づけるdictを作成"""
    # "O"のIDには0を割り当てる
    label2id = {"O": 0}
    # 固有表現タイプのsetを獲得して並び替える
    entity_types = set(
        [e["type"] for entities in entities_list for e in entities]
    )
    entity_types = sorted(entity_types)
    for i, entity_type in enumerate(entity_types):
        # "B-"のIDには奇数番号を割り当てる
        label2id[f"B-{entity_type}"] = i * 2 + 1
        # "I-"のIDには偶数番号を割り当てる
        label2id[f"I-{entity_type}"] = i * 2 + 2
    return label2id
        

In [26]:
# ラベルとIDを紐付けるdictを作成する
label2id = create_label2id(dataset["train"]["entities"])
id2label = {v: k for k, v in label2id.items()}
pprint(label2id)
pprint(id2label)

{'B-その他の組織名': 1,
 'B-イベント名': 3,
 'B-人名': 5,
 'B-地名': 7,
 'B-政治的組織名': 9,
 'B-施設名': 11,
 'B-法人名': 13,
 'B-製品名': 15,
 'I-その他の組織名': 2,
 'I-イベント名': 4,
 'I-人名': 6,
 'I-地名': 8,
 'I-政治的組織名': 10,
 'I-施設名': 12,
 'I-法人名': 14,
 'I-製品名': 16,
 'O': 0}
{0: 'O',
 1: 'B-その他の組織名',
 2: 'I-その他の組織名',
 3: 'B-イベント名',
 4: 'I-イベント名',
 5: 'B-人名',
 6: 'I-人名',
 7: 'B-地名',
 8: 'I-地名',
 9: 'B-政治的組織名',
 10: 'I-政治的組織名',
 11: 'B-施設名',
 12: 'I-施設名',
 13: 'B-法人名',
 14: 'I-法人名',
 15: 'B-製品名',
 16: 'I-製品名'}


#### データの前処理

In [27]:
from transformers.tokenization_utils_base import BatchEncoding

def preprocess_data(
    data: dict[str, Any],
    tokenizer: PreTrainedTokenizer,
    label2id: dict[int, str]
) -> BatchEncoding:
    """データの前処理"""
    # テキストのトークナイゼーションを行う
    inputs = tokenizer(
        data["text"],
        return_tensors="pt",
        return_special_tokens_mask=True,
    )
    inputs = {k: v.squeeze() for k, v in inputs.items()}

    # 文字のlistとトークンのlistのアライメントをとる
    characters = list(data["text"])
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
    char_to_token_indices, _ = get_alignments(characters, tokens)

    # "O"のIDのlistを作成する
    labels = torch.zeros_like(inputs["input_ids"])
    for entity in data["entities"]:  # 各固有表現を処理する
        start_token_indices = char_to_token_indices[entity["span"][0]]
        end_token_indices = char_to_token_indices[entity["span"][1]-1]
        # 文字に対応するトークンが存在しなければスキップする
        if (len(start_token_indices) == 0 or len(end_token_indices) == 0):
            continue
        start, end = start_token_indices[0], end_token_indices[0]
        entity_type = entity["type"]
        # 固有表現の開始トークンの位置に"B-"のIDを設定する
        labels[start] = label2id[f"B-{entity_type}"]
        # 固有表現の開始トークン以外の位置に"I-"のIDを設定する
        if start != end:
            labels[start+1:end+1] = label2id[f"I-{entity_type}"]
    # 特殊トークンの位置のIDは-100とする
    labels[torch.where(inputs["special_tokens_mask"])] = -100
    inputs["labels"] = labels
    return inputs

In [28]:
# 訓練セットに対して前処理を行う
train_dataset = dataset["train"].map(
    preprocess_data,
    fn_kwargs={
        "tokenizer": tokenizer,
        "label2id": label2id,
    },
    remove_columns=dataset["train"].column_names,
)
# 検証セットに対して前処理を行う
validation_dataset = dataset["validation"].map(
    preprocess_data,
    fn_kwargs={
        "tokenizer": tokenizer,
        "label2id": label2id,
    },
    remove_columns=dataset["validation"].column_names,
)

Parameter 'fn_kwargs'={'tokenizer': BertJapaneseTokenizer(name_or_path='cl-tohoku/bert-base-japanese-v3', vocab_size=32768, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}, 'label2id': {'O': 0, 'B-その他の組織名': 1, 'I-その他の組織名': 2, 'B-イベント名': 3, 'I-イベント名': 4

Map:   0%|          | 0/4274 [00:00<?, ? examples/s]

Map:   0%|          | 0/534 [00:00<?, ? examples/s]

#### モデルの準備

In [29]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

# モデルを読み込む
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    label2id=label2id,
    id2label=id2label
)

# パラメータをメモリ上に隣接した形で配置
# これを実行しない場合、モデルの保存でエラーになることがある
for param in model.parameters():
    param.data = param.data.contiguous()

# collate関数にDataCollatorForTokenClassificationを用いる
data_collator = DataCollatorForTokenClassification(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/447M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-v3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
pprint(label2id)

{'B-その他の組織名': 1,
 'B-イベント名': 3,
 'B-人名': 5,
 'B-地名': 7,
 'B-政治的組織名': 9,
 'B-施設名': 11,
 'B-法人名': 13,
 'B-製品名': 15,
 'I-その他の組織名': 2,
 'I-イベント名': 4,
 'I-人名': 6,
 'I-地名': 8,
 'I-政治的組織名': 10,
 'I-施設名': 12,
 'I-法人名': 14,
 'I-製品名': 16,
 'O': 0}


In [ ]:
pprint(id2label)

{0: 'O',
 1: 'B-その他の組織名',
 2: 'I-その他の組織名',
 3: 'B-イベント名',
 4: 'I-イベント名',
 5: 'B-人名',
 6: 'I-人名',
 7: 'B-地名',
 8: 'I-地名',
 9: 'B-政治的組織名',
 10: 'I-政治的組織名',
 11: 'B-施設名',
 12: 'I-施設名',
 13: 'B-法人名',
 14: 'I-法人名',
 15: 'B-製品名',
 16: 'I-製品名'}


#### モデルのファインチューニング

In [32]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import set_seed

# 乱数シードを42に固定する
set_seed(42)

# Trainerに渡す引数を初期化する
training_args = TrainingArguments(
    output_dir="output_bert_ner", # 結果の保存フォルダ
    per_device_train_batch_size=32, # 訓練時のバッチサイズ
    per_device_eval_batch_size=32, # 評価時のバッチサイズ
    learning_rate=1e-4, # 学習率
    lr_scheduler_type="linear", # 学習率スケジューラ
    warmup_ratio=0.1, # 学習率のウォームアップ
    num_train_epochs=5, # 訓練エポック数
    evaluation_strategy="epoch", # 評価タイミング
    save_strategy="epoch", # チェックポイントの保存タイミング
    logging_strategy="epoch", # ロギングのタイミング
    fp16=True, # 自動混合精度演算の有効化
    report_to="none",  # 外部ツールへのログを無効化
)

# Trainerを初期化する
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    args=training_args,
)

# 訓練する
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.676600,0.098428
2,0.068400,0.087866
3,0.029300,0.092028
4,0.013300,0.097189
5,0.006400,0.102355


TrainOutput(global_step=670, training_loss=0.15880151764670414, metrics={'train_runtime': 200.6938, 'train_samples_per_second': 106.481, 'train_steps_per_second': 3.338, 'total_flos': 1070012411245680.0, 'train_loss': 0.15880151764670414, 'epoch': 5.0})

In [33]:
pprint(train_dataset[0])

{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'input_ids': [2,
               16972,
               14284,
               384,
               50,
               13634,
               7075,
               20218,
               18124,
               7045,
               464,
               12913,
               385,
               3],
 'labels': [-100, 1, 2, 0, 1, 2, 2, 2, 2, 2, 0, 0, 0, -100],
 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


#### モデルの保存

##### Huging Face Hubへの保存

In [34]:
from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

login()

# Hugging Face Hubのリポジトリ名
repo_name = "tact-m/bert-base-japanese-v3-jnli"
# トークナイザとモデルをアップロード
tokenizer.push_to_hub(repo_name)
model.push_to_hub(repo_name)

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

### 6.3.2 固有表現の予測・抽出

#### 固有表現ラベルの予測

In [35]:
def convert_list_dict_to_dict_list(
    list_dict: dict[str, list]
) -> list[dict[str, list]]:
    """ミニバッチのデータを事例単位のlistに変換"""
    dict_list = []
    # dictのキーのlistを作成
    keys = list(list_dict.keys())
    for idx in range(len(list_dict[keys[0]])):  # 各事例で処理する
        # dictの各キーからデータを取り出してlistに追加する
        dict_list.append({key: list_dict[key][idx] for key in keys})
    return dict_list

In [36]:
# ミニバッチのデータを事例単位のlistに変換する
list_dict = {
    "input_ids": [[0,1], [2,3]],
    "labels": [[1,2], [3,4]]
}
dict_list = convert_list_dict_to_dict_list(list_dict)
print(f"input: {list_dict}")
print(f"output: {dict_list}")

input: {'input_ids': [[0, 1], [2, 3]], 'labels': [[1, 2], [3, 4]]}
output: [{'input_ids': [0, 1], 'labels': [1, 2]}, {'input_ids': [2, 3], 'labels': [3, 4]}]


In [37]:
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import PreTrainedModel

def run_prediction(
    dataloader: DataLoader,
    model: PreTrainedModel
) -> list[dict[str, Any]]:
    """予測スコアに基づき固有表現ラベルを予測"""
    predictions = []
    for batch in tqdm(dataloader):  # 各ミニバッチを処理する
        inputs = {
            k: v.to(model.device)
            for k, v in batch.items()
            if k != "special_tokens_mask"
        }
        # 予測スコアを取得する
        logits = model(**inputs).logits
        # 最もスコアの高いIDを取得する
        batch["pred_label_ids"] = logits.argmax(-1)
        batch = {k: v.cpu().tolist() for k, v in batch.items()}
        # ミニバッチのデータを事例単位のlistに変換する
        predictions += convert_list_dict_to_dict_list(batch)
    return predictions


In [38]:
# ミニバッチの作成にDataLoaderを用いる
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator,
)
# 固有表現ラベルを予測する
predictions = run_prediction(validation_dataloader, model)
print(predictions[0]["pred_label_ids"])

100%|██████████| 17/17 [00:00<00:00, 17.75it/s]

[0, 0, 15, 16, 0, 0, 13, 14, 14, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 13, 14, 14, 14, 13, 0, 0, 0, 0, 0, 15, 15, 15, 15, 13, 14, 14, 14, 16, 15, 13, 13, 14, 14, 0, 0, 0, 0, 13, 14, 14, 14, 13, 13, 13, 14, 14, 0, 15, 0, 0, 13, 0, 15, 15, 16, 0, 13, 14, 14, 14, 14, 13, 0, 0, 15, 15, 15, 16, 0, 13, 13, 14]


In [39]:
for pred_label_id in predictions[0]["pred_label_ids"]:
    print(id2label[pred_label_id])

O
O
B-製品名
I-製品名
O
O
B-法人名
I-法人名
I-法人名
O
O
O
O
O
O
O
O
O
B-製品名
B-法人名
I-法人名
I-法人名
I-法人名
B-法人名
O
O
O
O
O
B-製品名
B-製品名
B-製品名
B-製品名
B-法人名
I-法人名
I-法人名
I-法人名
I-製品名
B-製品名
B-法人名
B-法人名
I-法人名
I-法人名
O
O
O
O
B-法人名
I-法人名
I-法人名
I-法人名
B-法人名
B-法人名
B-法人名
I-法人名
I-法人名
O
B-製品名
O
O
B-法人名
O
B-製品名
B-製品名
I-製品名
O
B-法人名
I-法人名
I-法人名
I-法人名
I-法人名
B-法人名
O
O
B-製品名
B-製品名
B-製品名
I-製品名
O
B-法人名
B-法人名
I-法人名


#### 固有表現の抽出

In [40]:
from seqeval.metrics.sequence_labeling import get_entities

def extract_entities(
    predictions: list[dict[str, Any]],
    dataset: list[dict[str, Any]],
    tokenizer: PreTrainedTokenizer,
    id2label: dict[int, str]
) -> list[dict[str, Any]]:
    """固有表現を抽出"""
    results = []
    for prediction, data in zip(predictions, dataset):
        # 文字のlistを取得する
        characters = list(data["text"])

        # 特殊トークンを除いたトークンのlistと予測ラベルのlistを取得する
        tokens, pred_labels = [], []
        all_tokens = tokenizer.convert_ids_to_tokens(prediction["input_ids"])
        for token, label_id in zip(all_tokens, prediction["pred_label_ids"]):
            # 特殊トークン以外をlistに追加する
            if token not in tokenizer.all_special_tokens:
                tokens.append(token)
                pred_labels.append(id2label[label_id])
        
        # 文字のlistとトークンのlistのアライメントを取る
        _, token_to_char_indices = get_alignments(characters, tokens)
        
        # 予測ラベルのlistから固有表現タイプとトークン単位の開始位置と終了位置を取得して、
        # それらを正解データと同じ形式に変換する
        pred_entities = []
        for entity in get_entities(pred_labels):
            entity_type, token_start, token_end = entity
            # 文字単位の開始位置を取得する
            char_start = token_to_char_indices[token_start][0]
            # 文字単位の終了位置を取得する
            char_end = token_to_char_indices[token_end][-1] + 1
            pred_entity = {
                "name": "".join(characters[char_start:char_end]),
                "span": [char_start, char_end],
                "type": entity_type,
            }
            pred_entities.append(pred_entity)
        data["pred_entities"] = pred_entities
        results.append(data)
    return results

In [41]:
# 固有表現を抽出する
results = extract_entities(
    predictions, dataset["validation"], tokenizer, id2label
)
pprint(results[0:3])

[{'curid': '1662110',
  'entities': [{'name': '復活篇', 'span': [1, 4], 'type': '製品名'},
               {'name': 'グリーンバニー', 'span': [6, 13], 'type': '法人名'}],
  'pred_entities': [{'name': '復活篇', 'span': [1, 4], 'type': '製品名'},
                    {'name': 'グリーンバニー', 'span': [6, 13], 'type': '法人名'}],
  'text': '「復活篇」はグリーンバニーからの発売となっている。'},
 {'curid': '3024498',
  'entities': [{'name': '日刊ゲンダイ', 'span': [19, 25], 'type': '法人名'}],
  'pred_entities': [{'name': '日刊ゲンダイ', 'span': [19, 25], 'type': '法人名'}],
  'text': 'これらにより実質的な証拠調べが遅れたと日刊ゲンダイは報じている。'},
 {'curid': '3576262',
  'entities': [{'name': 'アンドリュー・スミス', 'span': [6, 16], 'type': '人名'}],
  'pred_entities': [{'name': 'アンドリュー・スミス', 'span': [6, 16], 'type': '人名'}],
  'text': 'プログラマのアンドリュー・スミスによれば、体の動きと頭の動きを独立させてしまうと、「カンニングできて」しまうパズルがあるという。'}]


### 6.3.3 検証セットを使用したモデルの選択

In [42]:
from glob import glob

best_score = 0
# 各チェックポイントで処理する
for checkpoint in sorted(glob("output_bert_ner/checkpoint-*")):
    # print(checkpoint)
    # モデルを読み込む
    model = AutoModelForTokenClassification.from_pretrained(
        checkpoint
    )
    model.to("cuda:0")
    # 固有表現ラベルを予測する
    predictions = run_prediction(validation_dataloader, model)
    # 固有表現ラベルを抽出する
    results = extract_entities(
        predictions,
        dataset["validation"],
        tokenizer,
        id2label
    )
    # 正解データと予測データのラベルのlistを作成する
    true_labels, pred_labels = convert_results_to_labels(results)
    # 評価スコアを算出する
    scores = compute_scores(true_labels, pred_labels, "micro")
    if best_score < scores["f1-score"]:
        best_score = scores["f1-score"]
        best_model = model

100%|██████████| 17/17 [00:03<00:00,  4.97it/s]


### 6.3.4 性能評価

In [43]:
# テストセットに対して前処理を行う
test_dataset = dataset["test"].map(
    preprocess_data,
    fn_kwargs = {
        "tokenizer": tokenizer,
        "label2id": label2id,
    },
    remove_columns =dataset["test"].column_names,
)
# ミニバッチの作成にDataLoaderを用いる
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator,
)
# 固有表現ラベルを予測する
predictions = run_prediction(test_dataloader, best_model)
# 固有表現を抽出する
results = extract_entities(
    predictions,
    dataset["test"],
    tokenizer,
    id2label
)
# 正解データと予測データのラベルlistを作成する
true_labels, pred_labels = convert_results_to_labels(results)
# 評価結果を出力する
print(classification_report(true_labels, pred_labels))

Map:   0%|          | 0/535 [00:00<?, ? examples/s]

100%|██████████| 17/17 [00:03<00:00,  4.50it/s]


              precision    recall  f1-score   support

     その他の組織名       0.76      0.83      0.79       100
       イベント名       0.84      0.92      0.88        93
          人名       0.94      0.95      0.95       287
          地名       0.88      0.86      0.87       204
      政治的組織名       0.77      0.87      0.82       106
         施設名       0.82      0.85      0.83       137
         法人名       0.87      0.87      0.87       248
         製品名       0.78      0.83      0.80       158

   micro avg       0.85      0.88      0.87      1333
   macro avg       0.83      0.87      0.85      1333
weighted avg       0.85      0.88      0.87      1333



### 6.3.5 エラー分析

In [44]:
def find_error_results(
    results: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """エラー事例を発見"""
    error_results = []
    for idx, result in enumerate(results):  # 各事例を処理する
        result["idx"] = idx
        # 正解データと予測データが異なるならばlistに加える
        if result["entities"] != result["pred_entities"]:
            error_results.append(result)
    return error_results

def output_text_with_label(
    result: dict[str, Any],
    entity_column: str,
) -> str:
    """固有表現ラベル付きテキストを出力"""
    text_with_label = ""
    entity_count = 0
    for i, char in enumerate(result["text"]):  # 各文字を処理する
        # 出力に加えていない固有表現の有無を判定する
        if entity_count < len(result[entity_column]):
            entity = result[entity_column][entity_count]
            # 固有表現の先頭の処理を行う
            if i == entity["span"][0]:
                entity_type = entity["type"]
                text_with_label += f" [({entity_type})] "
            text_with_label += char
            # 固有表現の末尾の処理を行う
            if i == entity["span"][1] - 1:
                text_with_label += "] "
                entity_count += 1
        else:
            text_with_label += char
    return text_with_label

In [45]:
# エラー事例を発見する
error_results = find_error_results(results)
# 3件のエラー事例を出力する
for result in error_results[:5]:
    idx = result["idx"]
    true_text = output_text_with_label(result, "entities")
    pred_text = output_text_with_label(result, "pred_entities")
    print(f"事例{idx}の正解: {true_text}")
    print(f"事例{idx}の予測: {pred_text}")
    print()

事例11の正解: このため、本形式は [(法人名)] 江ノ島電鉄] では電動貨車2号車と呼称されているほか、電貨2と呼称される場合もある。
事例11の予測: このため、本形式は [(法人名)] 江ノ島電鉄] では電動貨車2号車と呼称されているほか、 [(製品名)] 電貨] 2と呼称される場合もある。

事例16の正解: 「 [(製品名)] 兵動・野爆のキャンピング王国] 」とのコラボレーション番組。
事例16の予測: 「 [(人名)] 兵動]  [(製品名)] ・]  [(人名)] 野爆]  [(製品名)] のキャンピング王国] 」とのコラボレーション番組。

事例18の正解:  [(法人名)] 常盤木学園] 時代の同級生に [(その他の組織名)] なでしこジャパン] の [(人名)] 熊谷紗希] がいる。
事例18の予測:  [(施設名)] 常盤木学園] 時代の同級生に [(その他の組織名)] なでしこジャパン] の [(人名)] 熊谷紗希] がいる。

事例19の正解: テレビで狼男映画の「 [(製品名)] 倫敦の人狼] 」を見た [(人名)] フィル・エヴァリー] は「ロンドンの狼男というタイトルで踊り騒げる曲を書いてみないか」と [(法人名)] ジヴォン] に持ちかけた。
事例19の予測: テレビで狼男映画の「 [(製品名)] 倫敦の人狼] 」を見た [(人名)] フィル・エヴァリー] は「 [(製品名)] ロンドンの狼男] というタイトルで踊り騒げる曲を書いてみないか」と [(人名)] ジヴォン] に持ちかけた。

事例27の正解:  [(政治的組織名)] 李承晩政権] 期から [(政治的組織名)] 朴正煕政権] 期の1970年前後まで、南側の [(地名)] 大韓民国] よりも北側の [(地名)] 朝鮮民主主義人民共和国] の方が経済的な体力では勝っていたのである。
事例27の予測:  [(人名)] 李承晩]  [(政治的組織名)] 政権] 期から [(人名)] 朴正煕] 政権期の1970年前後まで、南側の [(地名)] 大韓民国] よりも北側の [(地名)] 朝鮮民主主義人民共和国] の方が経済的な体力では勝っていたのである。

